[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedorShind/EMRG/blob/main/docs/tutorials/vqe_h2_mitigation.ipynb)

# Mitigating VQE Errors on H2 with EMRG

EMRG analyzes a quantum circuit and generates ready-to-run Mitiq error mitigation code. It can select ZNE, PEC, CDR, or a composite ZNE-over-PEC recipe based on circuit depth, gate mix, noise-model availability, and estimated sampling cost. This notebook builds a 4-qubit H2 VQE ansatz and shows how each technique changes the generated recipe.

In [ ]:
!pip install -q emrg[preview]

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

from emrg import analyze_circuit, generate_recipe

## Build the H2 VQE ansatz

Hardware-efficient ansatz: 4 qubits, 2 layers of RY rotations and CNOT entanglement in the STO-3G basis. Parameters are bound to fixed values for demonstration. This ansatz structure follows patterns identified in autonomous VQE architecture search ([github.com/FedorShind/autoresearch-qc](https://github.com/FedorShind/autoresearch-qc)).

In [ ]:
rng = np.random.default_rng(42)
n_qubits = 4
n_layers = 2

qc = QuantumCircuit(n_qubits, n_qubits)
for _ in range(n_layers):
    for q in range(n_qubits):
        qc.ry(float(rng.uniform(0, 2 * np.pi)), q)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
qc.measure(range(n_qubits), range(n_qubits))

print(f"Qubits: {qc.num_qubits}")
print(f"Depth:  {qc.depth()}")
qc.draw(output="text")

## Analyze the circuit

`analyze_circuit` extracts features from the circuit using static analysis only; no simulation.

In [ ]:
features = analyze_circuit(qc)

print(f"Depth:               {features.depth}")
print(f"Total gates:         {features.total_gate_count}")
print(f"Multi-qubit gates:   {features.multi_qubit_gate_count}")
print(f"Noise estimate:      {features.estimated_noise_factor}")
print(f"Noise category:      {features.noise_category}")
print(f"Non-Clifford count:  {features.non_clifford_count}")
print(f"Non-Clifford frac:   {features.non_clifford_fraction:.4f}")
print(f"PEC overhead est:    {features.pec_overhead_estimate:.2f}")
print(f"Layer heterogeneity: {features.layer_heterogeneity:.4f}")

The RY gates with arbitrary angles are non-Clifford (57% of total gates). The depth of 8 falls below CDR's minimum depth threshold of 10, so EMRG routes this circuit to ZNE by default. The PEC overhead of about 2.3x is low enough for PEC when a noise model is available. Composite is not automatic here because the circuit is too shallow; adding ZNE-over-PEC would spend extra samples without a strong reason.

## Generate the default recipe

`generate_recipe` chains analysis, technique selection, and code generation in a single call.

In [ ]:
result = generate_recipe(qc, explain=True)

print(f"Technique:     {result.recipe.technique}")
print(f"Factory:       {result.recipe.factory_name}")
print(f"Scale factors: {result.recipe.scale_factors}")
print(f"Scaling:       {result.recipe.scaling_method}")
print()
print(result.code)

EMRG selected ZNE with LinearFactory because the depth (8) is below 20 and the multi-qubit gate count (6) is low. This is the conservative ZNE configuration: three scale factors, linear extrapolation, and global folding.

## PEC recipe

PEC (Probabilistic Error Cancellation) provides unbiased error correction at the cost of exponential sampling overhead. It requires a noise model. Setting `noise_model_available=True` enables PEC consideration.

In [ ]:
pec_result = generate_recipe(qc, noise_model_available=True, explain=True)

print(f"Technique: {pec_result.recipe.technique}")
print(f"Overhead:  {pec_result.features.pec_overhead_estimate:.2f}x")
print()
for line in pec_result.rationale:
    print(f"  - {line}")

EMRG selected PEC here because the circuit is shallow (depth 8, below the PEC threshold of 30) and the overhead is about 2.3x, well below the 1000x cutoff. PEC has higher priority than CDR or ZNE when its conditions are met.

## Composite recipe

Composite mitigation runs ZNE outside PEC: ZNE creates noise-scaled circuits, and PEC corrects each scaled circuit before extrapolation. It is meant for moderate-depth, PEC-eligible circuits where a noise model is available and plain PEC or plain ZNE leaves too much bias. This H2 circuit is too shallow for automatic composite selection, so the cell below forces composite only to show the generated shape.

In [ ]:
composite_result = generate_recipe(
    qc,
    technique="composite",
    noise_model_available=True,
    explain=True,
)

print(f"Technique: {composite_result.recipe.technique}")
print(f"Overhead:  {composite_result.recipe.estimated_overhead:.1f}x")
print("Components:")
for component in composite_result.recipe.components:
    if component.technique == "zne":
        detail = f"{component.factory_name}, scales {component.scale_factors}"
    elif component.technique == "pec":
        detail = f"{component.factory_kwargs.get('num_samples')} samples"
    else:
        detail = ""
    print(f"  - {component.technique.upper()}: {detail}")
print()
for line in composite_result.rationale:
    print(f"  - {line}")

For this circuit, composite is an educational comparison rather than the recommended default. The useful takeaway is the execution order: PEC becomes the executor inside the outer ZNE call.

## CDR recipe

CDR (Clifford Data Regression) replaces non-Clifford gates with Clifford substitutes to create training circuits that can be simulated classically. A regression model fitted on these training results corrects the noisy output. CDR sits between ZNE (approximate, no noise model needed) and PEC (exact, expensive) in terms of accuracy and cost.

In [ ]:
cdr_result = generate_recipe(qc, technique="cdr")

kwargs = cdr_result.recipe.factory_kwargs
print(f"Technique:          {cdr_result.recipe.technique}")
print(f"Training circuits:  {kwargs.get('num_training_circuits')}")
print(f"Fit method:         {kwargs.get('fit_method')}")
print()
print(cdr_result.code)

CDR was forced here with `technique="cdr"`. EMRG would not auto-select CDR for this circuit because the depth (8) is below CDR's minimum threshold of 10. For circuits with depth 10 to 40 and more than 20% non-Clifford gates, CDR is selected automatically.

## Preview mode

`preview=True` runs a noisy simulation (Cirq DensityMatrixSimulator with depolarizing noise) and compares ideal, noisy, and mitigated expectation values. Requires `pip install emrg[preview]`.

In [ ]:
preview_result = generate_recipe(qc, preview=True, noise_level=0.01)

if preview_result.preview and preview_result.preview.ideal_value is not None:
    p = preview_result.preview
    print(f"Technique:  {p.technique}")
    print(f"Ideal:      {p.ideal_value:+.4f}")
    print(f"Noisy:      {p.noisy_value:+.4f}  (error: {p.noisy_error:.4f})")
    print(f"Mitigated:  {p.mitigated_value:+.4f}  (error: {p.mitigated_error:.4f})")
    print(f"Reduction:  {p.error_reduction:.1f}x")
else:
    p = preview_result.preview
    warning = p.warning if p else "cirq not installed"
    print(f"Preview skipped: {warning}")

## Summary

EMRG analyzed a 4-qubit H2 VQE circuit and selected ZNE (LinearFactory) as the default recipe, PEC when a noise model is available, and CDR or composite when forced. The techniques cover different tradeoffs: ZNE is approximate and broadly available, PEC is unbiased but sample-hungry, CDR targets non-Clifford-heavy circuits, and composite ZNE-over-PEC is reserved for moderate-depth circuits where both extrapolation and known-noise cancellation are worth the cost.

- [EMRG on GitHub](https://github.com/FedorShind/EMRG)
- [EMRG on PyPI](https://pypi.org/project/emrg/)
- [Mitiq documentation](https://mitiq.readthedocs.io/)